In [6]:
import os
import pandas as pd
from pysr import PySRRegressor
from sympy import cos  # Import SymPy for custom operator definitions
import re
elements = ['Al', 'Zn', 'Mn', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']
def extract_total_solute(label):
    total = 0.0
    for el in elements:
        match = re.search(rf'([\d.]+){el}', label)
        if match:
            total += float(match.group(1))
    return round(total, 3)
def run_experiment(task_name, X, Y, data_file, target_name, base_directory="./"):
    # Ensure the base directory exists and navigate to it
    os.makedirs(base_directory, exist_ok=True)

    # Create a folder for the task within the base directory
    task_folder = os.path.join(base_directory, task_name)
    
    # Ensure that the task folder is created, without nesting
    os.makedirs(task_folder, exist_ok=True)
    
    # Change the working directory to the task folder
    os.chdir(task_folder)
    print(f"Working directory changed to: {os.getcwd()}")


    # Initialize the PySR model with customized settings
    model = PySRRegressor(
        maxsize=50,  # Maximum complexity
        niterations=100,  # Number of iterations
        binary_operators=["*", "+", "-", "/"],
        unary_operators=["square", "cube", "exp", "cos2(x)=cos(x)^2"],
        constraints={
            "/": (-1, 9),
            "square": 9,
            "cube": 9,
            "exp": 9,
        },
        nested_constraints={
            "square": {"square": 1, "cube": 1, "exp": 0},
            "cube": {"square": 1, "cube": 1, "exp": 0},
            "exp": {"square": 1, "cube": 1, "exp": 0},
        },
        extra_sympy_mappings={
        "inv": lambda x: 1 / x,  # Inverse operator
        "cos2": lambda x: cos(x)**2,  # Custom operator for `cos2`
    },       
        elementwise_loss="loss(prediction, target) = (prediction - target)^2",
        early_stop_condition=(
            "stop_if(loss, complexity) = loss < 1e-6 && complexity < 10"
        ),
        timeout_in_seconds=60 * 60 * 2,  # Timeout in seconds
    )

    # Fit the model and save the results in the task folder
    print(f"Starting experiment for task: {task_name}")
    model.fit(X, Y)
    print(f"Experiment {task_name} complete. Results saved to: {task_folder}")

def get_dataset_qf(data_file):
    Target = '屈服强度'
    names = ['Shear modulus mean', 'Grain Size', 'MagpieData range GSvolume_pa', 'Mixing enthalpy','Solute content']
    Feature = names
    df = pd.read_excel(data_file, header=0)
    df['Solute content'] = df['追踪编号'].apply(extract_total_solute)
    X = df[Feature]  # Ensure column headers are correctly read
    Y = pd.read_excel(data_file, header=0)[Target]
    return X, Y

def get_dataset_kl(data_file):
    Target = '抗拉强度 (UTS)'
    names = ['Shear modulus mean', 'Grain Size', 'MagpieData range GSvolume_pa', 'Mixing enthalpy','Solute content']
    Feature = names
    df = pd.read_excel(data_file, header=0)
    df['Solute content'] = df['追踪编号'].apply(extract_total_solute)
    X = df[Feature]  # Ensure column headers are correctly read
    Y = pd.read_excel(data_file, header=0)[Target]
    return X, Y

# Task and dataset configuration
train_data_file_qf = r'G:\AIMS_Lab_final/my_project\AAA-MgAlloy\AI_Alloy/Yield_Strength/train_set_new.xlsx'
train_data_file_kl = r'G:\AIMS_Lab_final/my_project\AAA-MgAlloy\AI_Alloy/Tensile_Strength/train_set_new.xlsx'


# Run experiments for QF dataset

X_qf, Y_qf = get_dataset_qf(train_data_file_qf)
run_experiment(f"qf", X_qf, Y_qf, train_data_file_qf, '屈服强度')

# Run experiments for KL dataset

X_kl, Y_kl = get_dataset_kl(train_data_file_kl)
run_experiment(f"kl", X_kl, Y_kl, train_data_file_kl, '抗拉强度 (UTS)')


Working directory changed to: g:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\symbolic_regression\GPSR\qf
Starting experiment for task: qf
Compiling Julia backend...


d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:1404: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:2727: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:1529: UserWarning: Spaces in DataFrame column names are not supported. Spaces have been replaced with underscores. 
Please rename the columns to valid names.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 2.370e+05
Progress: 1102 / 3100 total iterations (35.548%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           7.078e+03  1.594e+01  y = 188.68
3           6.887e+03  1.368e-02  y = Solute_content + 180.48
5           5.054e+03  1.548e-01  y = 22910 / (Grain_Size - -90.906)
7           4.690e+03  3.729e-02  y = (2365.3 / (Grain_Size - -12.647)) + 104.69
9           4.341e+03  3.872e-02  y = (320.46 * (Solute_content + 43.543)) / (Grain_Size - -...
                                      62.133)
11          3.624e+03  9.024e-02  y = (((Solute_content + 6.997) / (Grain_Size - -12.039)) +...
                                       0.66028) * 164.84
13          3.442e+03  2.584e-02  y = (Solute_content + 17.065) * ((110.42 / (Grain_Size - -...
              

[ Info: Final population:
[ Info: Results saved to:


Working directory changed to: g:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\symbolic_regression\GPSR\qf\kl
Starting experiment for task: kl


d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:1404: UserWarning: Note: Using a large maxsize for the equation search will be exponentially slower and use significant memory.
  warnings.warn(
d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:2727: UserWarning: Note: it looks like you are running in Jupyter. The progress bar will be turned off.
  warnings.warn(
d:\Anaconda\envs\Mg_Alloy\lib\site-packages\pysr\sr.py:1529: UserWarning: Spaces in DataFrame column names are not supported. Spaces have been replaced with underscores. 
Please rename the columns to valid names.
  warnings.warn(
[ Info: Started!



Expressions evaluated per second: 1.570e+05
Progress: 750 / 3100 total iterations (24.194%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.355e+03  1.594e+01  y = 275.84
3           6.254e+03  7.963e-03  y = Solute_content + 267.64
5           5.598e+03  5.539e-02  y = (673.86 - Grain_Size) * 0.44047
7           5.536e+03  5.616e-03  y = 292.9 - ((Grain_Size - Solute_content) * 0.40838)
9           5.062e+03  4.473e-02  y = (249.5 - (-504.35 / (Grain_Size + 2.5365))) + -9.3788
11          4.693e+03  3.784e-02  y = Solute_content - ((-25.8 - (-613.62 / (-25.294 - Grain...
                                      _Size))) * 6.811)
13          4.495e+03  2.157e-02  y = (Solute_content - ((25.33 / ((-17.579 - Grain_Size) / ...
                                      29.727)) + -48.238))

[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 1.590e+05
Progress: 2258 / 3100 total iterations (72.839%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.355e+03  1.594e+01  y = 275.84
3           6.254e+03  7.963e-03  y = Solute_content + 267.64
5           5.587e+03  5.639e-02  y = (Grain_Size * -0.39523) + 295.59
7           4.806e+03  7.535e-02  y = (-2342.1 / (-16.272 - Grain_Size)) + 211.29
9           4.712e+03  9.862e-03  y = ((-3320.8 / (-22.582 - Grain_Size)) + 201.43) + Solute...
                                      _content
11          4.457e+03  2.783e-02  y = (Solute_content - (-47.788 - (-899.82 / (-20.425 - Gra...
                                      in_Size)))) * 3.5582
13          4.162e+03  3.417e-02  y = (6.2606 - Mixing_enthalpy) * (Solute_content - ((613.8...
        

In [1]:
import pandas as pd
df = pd.read_excel('G:\AIMS_Lab_final/my_project\AAA-MgAlloy\AI_Alloy/Yield_Strength/train_set_new.xlsx'
)
df.columns

Index(['Unnamed: 0', 'MagpieData minimum Number', 'MagpieData maximum Number',
       'MagpieData range Number', 'MagpieData mean Number',
       'MagpieData avg_dev Number', 'MagpieData mode Number',
       'MagpieData minimum MendeleevNumber',
       'MagpieData maximum MendeleevNumber',
       'MagpieData range MendeleevNumber',
       ...
       'frac f valence electrons', 'Grain Size', 'Precipitate Distribution',
       'Habit Plane', 'Calculated Solid Solution', 'Calculated Grain Boundary',
       'Calculated Yield Strength', '屈服强度', '抗拉强度 (UTS)', '追踪编号'],
      dtype='object', length=282)

In [9]:
import re

# 模拟从文本中读取的每一行数据
lines = [
    "1,7077.953",
    "3,6886.96",
    "5,5053.5728",
    "7,4624.1953",
    "9,3912.6238",
    "11,3607.8264",
    "13,3441.173",
    "14,3339.7432",
    "16,3332.1895",
    "17,3320.9893",
    "19,3274.0203",
    "21,3267.3015",
    "22,3248.9666",
    "25,3243.3499",
    "27,3215.481",
    "28,3206.9717",
    "29,3205.4983",
    "31,3204.7808",
    "32,3202.0684",
    "35,3175.9814",
    "37,3175.7563",
    "40,3166.605",
    "42,3121.3481",
    "44,3119.957"
]

# 提取并打印
for i, line in enumerate(lines, start=1):
    parts = line.split(",")
    if len(parts) == 2:
        complexity = int(parts[0])
        loss = float(parts[1])
        print(f" {loss}")

lines = [
    "1,6354.6426",
    "3,6254.2383",
    "5,5587.2427",
    "7,4804.389",
    "9,4642.643",
    "11,4456.5073",
    "13,4162.241",
    "15,4116.2944",
    "16,3930.145",
    "18,3913.7124",
    "20,3896.0957",
    "22,3895.2378",
    "23,3846.602",
    "25,3820.0967",
    "27,3818.3782",
    "28,3817.1765",
    "29,3766.6511",
    "30,3729.7383",
    "32,3718.9788",
    "36,3696.4465",
    "38,3649.6885",
    "40,3637.688",
    "42,3623.5737",
    "43,3620.1526",
    "44,3615.5447",
    "45,3611.748"
]
print('xxxxxxx')
# 提取并打印行号和 loss 值
for i, line in enumerate(lines, start=1):
    parts = line.split(",")
    if len(parts) == 2:
        complexity = int(parts[0])
        loss = float(parts[1])
        print(f" {loss}")

 7077.953
 6886.96
 5053.5728
 4624.1953
 3912.6238
 3607.8264
 3441.173
 3339.7432
 3332.1895
 3320.9893
 3274.0203
 3267.3015
 3248.9666
 3243.3499
 3215.481
 3206.9717
 3205.4983
 3204.7808
 3202.0684
 3175.9814
 3175.7563
 3166.605
 3121.3481
 3119.957
xxxxxxx
 6354.6426
 6254.2383
 5587.2427
 4804.389
 4642.643
 4456.5073
 4162.241
 4116.2944
 3930.145
 3913.7124
 3896.0957
 3895.2378
 3846.602
 3820.0967
 3818.3782
 3817.1765
 3766.6511
 3729.7383
 3718.9788
 3696.4465
 3649.6885
 3637.688
 3623.5737
 3620.1526
 3615.5447
 3611.748


In [12]:
# 图1：Complexity 和 Loss
loss_dict_1 = {
    1: 7077.953, 3: 6886.96, 5: 5053.5728, 7: 4624.1953, 9: 3912.6238, 11: 3607.8264,
    13: 3441.173, 14: 3339.7432, 16: 3332.1895, 17: 3320.9893, 19: 3274.0203,
    21: 3267.3015, 22: 3248.9666, 25: 3243.3499, 27: 3215.481, 28: 3206.9717,
    29: 3205.4983, 31: 3204.7808, 32: 3202.0684, 35: 3175.9814, 37: 3175.7563,
    40: 3166.605, 42: 3121.3481, 44: 3119.957
}

# 图2：Complexity 和 Loss
loss_dict_2 = {
    1: 6354.6426, 3: 6254.2383, 5: 5587.2427, 7: 4804.389, 9: 4642.643, 11: 4456.5073,
    13: 4162.241, 15: 4116.2944, 16: 3930.145, 18: 3913.7124, 20: 3896.0957, 22: 3895.2378,
    23: 3846.602, 25: 3820.0967, 27: 3818.3782, 28: 3817.1765, 29: 3766.6511, 30: 3729.7383,
    32: 3718.9788, 36: 3696.4465, 38: 3649.6885, 40: 3637.688, 42: 3623.5737, 43: 3620.1526,
    44: 3615.5447, 45: 3611.748
}

# 取交集 ID
common_ids = sorted(set(loss_dict_1.keys()) & set(loss_dict_2.keys()))

# 整理为表格形式
import pandas as pd

data = {
    "Complexity": common_ids,
    "Loss_1": [loss_dict_1[i] for i in common_ids],
    "Loss_2": [loss_dict_2[i] for i in common_ids]
}

df_common = pd.DataFrame(data)
print(df_common)
df_common.to_excel("common_loss_comparison.xlsx", index=False)



    Complexity     Loss_1     Loss_2
0            1  7077.9530  6354.6426
1            3  6886.9600  6254.2383
2            5  5053.5728  5587.2427
3            7  4624.1953  4804.3890
4            9  3912.6238  4642.6430
5           11  3607.8264  4456.5073
6           13  3441.1730  4162.2410
7           16  3332.1895  3930.1450
8           22  3248.9666  3895.2378
9           25  3243.3499  3820.0967
10          27  3215.4810  3818.3782
11          28  3206.9717  3817.1765
12          29  3205.4983  3766.6511
13          32  3202.0684  3718.9788
14          40  3166.6050  3637.6880
15          42  3121.3481  3623.5737
16          44  3119.9570  3615.5447


In [ ]:

# qf
((Shear_modulus_mean / (Grain_Size * square(Grain_Size))) + ((11.649739 + Solute_content) * (square((48.917896 / (Grain_Size - -27.711927)) + (2.184571 - Mixing_enthalpy)) + (3.9850383 + cube(cos2(Mixing_enthalpy - square(MagpieData_range_GSvolume_pa / 1.9727559))))))) - ((cos2(Mixing_enthalpy + square(MagpieData_range_GSvolume_pa)) * Shear_modulus_mean) + (8.995719 + Shear_modulus_mean))

# kl
(((-0.5954335 - (Mixing_enthalpy * 131.96585)) + ((-158.72464 / (1.1267804 + (Mixing_enthalpy * MagpieData_range_GSvolume_pa))) - (13.028601 * ((Shear_modulus_mean / ((-23.958717 - Grain_Size) / (Solute_content + 6.7804537))) + -18.068134)))) + (Shear_modulus_mean + (-1.7549324 + (((Solute_content * 3.3655095) + Shear_modulus_mean) * cos2(MagpieData_range_GSvolume_pa))))) + (MagpieData_range_GSvolume_pa * Mixing_enthalpy)

In [1]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.metrics import r2_score, mean_absolute_error

# ---------- Step 1: 元素提取 ----------
elements = ['Al', 'Zn', 'Mn', 'Li', 'Sn', 'Ce', 'La', 'Ca', 'Si', 'Cu', 'Ni',
            'Fe', 'Bi', 'Ti', 'Sr', 'Sm', 'Sc', 'Gd', 'Y', 'Zr', 'Nd', 'Ag', 'Yb']

def extract_total_solute(label):
    total = 0.0
    for el in elements:
        match = re.search(rf'([\d.]+){el}', label)
        if match:
            total += float(match.group(1))
    return round(total, 3)

# ---------- Step 2: 数学函数 ----------
def square(x): 
    return x ** 2

def cube(x): 
    return x ** 3

def cos2(x): 
    return np.cos(x) ** 2

# ---------- Step 3: 符号回归公式 ----------
def predict_qf(row):
    g = row['Grain Size']
    s = row['Shear modulus mean']
    v = row['MagpieData range GSvolume_pa']
    m = row['Mixing enthalpy']
    c = row['Solute content']
    
    term1 = s / (g * square(g))
    inner1 = 48.917896 / (g + 27.711927)
    term2 = (11.649739 + c) * (square(inner1 + (2.184571 - m)) + (3.9850383 + cube(cos2(m - square(v / 1.9727559)))))
    term3 = cos2(m + square(v)) * s
    term4 = 8.995719 + s
    return term1 + term2 - (term3 + term4)

def predict_kl(row):
    g = row['Grain Size']
    s = row['Shear modulus mean']
    v = row['MagpieData range GSvolume_pa']
    m = row['Mixing enthalpy']
    c = row['Solute content']
    
    part1 = -0.5954335 - (m * 131.96585)
    part2 = -158.72464 / (1.1267804 + (m * v))
    part3 = 13.028601 * ((s / ((-23.958717 - g) / (c + 6.7804537))) + -18.068134)
    part4 = s + (-1.7549324 + ((c * 3.3655095 + s) * cos2(v)))
    part5 = v * m
    return part1 + (part2 - part3) + part4 + part5

# ---------- Step 4: 读取数据 ----------
train_data_file_qf = r'G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\Yield_Strength\train_set_new.xlsx'
train_data_file_kl = r'G:\AIMS_Lab_final\my_project\AAA-MgAlloy\AI_Alloy\Tensile_Strength\train_set_new.xlsx'

df_qf = pd.read_excel(train_data_file_qf)
df_kl = pd.read_excel(train_data_file_kl)

df_qf['Solute content'] = df_qf['追踪编号'].apply(extract_total_solute)
df_kl['Solute content'] = df_kl['追踪编号'].apply(extract_total_solute)

# ---------- Step 5: 预测 ----------
df_qf['qf_pred'] = df_qf.apply(predict_qf, axis=1)
df_kl['kl_pred'] = df_kl.apply(predict_kl, axis=1)

# ---------- Step 6: 评估（含 MAPE） ----------
# 屈服强度
y_true_qf = df_qf['屈服强度'].values
y_pred_qf = df_qf['qf_pred'].values

r2_qf = r2_score(y_true_qf, y_pred_qf)
mae_qf = mean_absolute_error(y_true_qf, y_pred_qf)
# 避免除以 0，先对零值做简单处理（如果数据里没有 0，就不需要这步）
epsilon = 1e-8
mape_qf = np.mean(np.abs((y_true_qf - y_pred_qf) / (np.where(y_true_qf == 0, epsilon, y_true_qf)))) * 100

# 抗拉强度
y_true_kl = df_kl['抗拉强度 (UTS)'].values
y_pred_kl = df_kl['kl_pred'].values

r2_kl = r2_score(y_true_kl, y_pred_kl)
mae_kl = mean_absolute_error(y_true_kl, y_pred_kl)
mape_kl = np.mean(np.abs((y_true_kl - y_pred_kl) / (np.where(y_true_kl == 0, epsilon, y_true_kl)))) * 100

print("✅ 屈服强度:")
print(f"R²  = {r2_qf:.4f}")
print(f"MAE = {mae_qf:.4f}")
print(f"MAPE = {mape_qf:.2f}%")

print("\n✅ 抗拉强度:")
print(f"R²  = {r2_kl:.4f}")
print(f"MAE = {mae_kl:.4f}")
print(f"MAPE = {mape_kl:.2f}%")

# ---------- Step 7: 可选保存 ----------
df_qf.to_excel("qf_formula_predictions.xlsx", index=False)
df_kl.to_excel("kl_formula_predictions.xlsx", index=False)


✅ 屈服强度:
R²  = 0.5592
MAE = 42.0098
MAPE = 28.03%

✅ 抗拉强度:
R²  = 0.4310
MAE = 46.8144
MAPE = 19.02%
